# Kafka and Spark on Colab, talking to KNIME on your laptop

## Option B: no account needed

### BAN 5600, Clark University

This notebook turns a free Google Colab machine into a small **live text analysis
service**. KNIME, running on your own laptop, sends it a sentence. A few seconds later
KNIME gets back a verdict: is that sentence positive, negative or neutral, and which
words decided it.

You do not need to know Kafka, Spark or networking to run this. Follow the cells in
order, top to bottom, and read the short explanation above each one.

---

## What you are building

```
   YOUR LAPTOP                    THE INTERNET                 GOOGLE COLAB
  +--------------+                                      +----------------------+
  |              |                                      |                      |
  |   KNIME      |   1. sends "I love this product"     |   KAFKA              |
  |   Producer   | -----------------------------------> |   topic: knime-text  |
  |              |              a tunnel                |         |            |
  |              |                                      |         v            |
  |              |                                      |   SPARK              |
  |              |                                      |   counts positive    |
  |              |                                      |   and negative words |
  |              |                                      |         |            |
  |   KNIME      |   4. receives POSITIVE, score +1.00  |         v            |
  |   Consumer   | <----------------------------------- |   KAFKA              |
  |              |                                      |   knime-sentiment    |
  +--------------+                                      +----------------------+
```

**Kafka** is the post office in the middle. It holds messages in named mailboxes called
**topics**. Anyone can drop a message in; anyone can read it.

**Spark** is the worker. It watches the `knime-text` topic, scores every sentence that
arrives, and drops the answer into the `knime-sentiment` topic.

**KNIME** never talks to Spark directly. It only ever talks to Kafka. That is the point
of this design: the two sides do not need to know anything about each other.

---

## Why a tunnel is needed, in one paragraph

Colab machines can reach out to the internet, but nothing on the internet can reach
**in** to them. There is no address you can type that lands on your Colab machine. A
**tunnel** fixes that: a small program on Colab opens a connection outwards to a public
server, and that server lends you a public address. Anything sent to the public address
travels back down the already open connection into Colab. KNIME connects to the public
address and never knows the difference.

---

## Before you start: nothing

There is no account to create, no token to paste and no credit card. The tunnel
this notebook uses, **bore**, is a single small program on a free shared server.

The trade is reliability. `bore.pub` is run by one person as a free public
service, with no guarantees. It is usually fine and the round trip measures about
a quarter of a second, but it can be busy. If it fails twice in a row, open
**`kafka_knime_bridge_A_ngrok.ipynb`** instead, which uses ngrok. That one wants a
free account and a card for verification, and in exchange it is more dependable.

Everything from Step 3 onwards is identical in both notebooks.


---

# Step 1. Install Java, Kafka and Spark

Kafka and Spark are both written in Java, so Java has to go on first. Then we download
Kafka itself, then Spark's Python interface.

This cell takes **about two minutes** the first time. It prints a tick for each piece as
it finishes. You only need to run it once per Colab session.

**About the download address.** Apache keeps current releases on `dlcdn.apache.org`, which
is fast, and older ones on `archive.apache.org`, which is throttled to about 120 KB/s. At
that speed the Kafka download alone takes seventeen minutes and looks frozen. So this cell
asks dlcdn which version it currently has, takes that one, and only falls back to the slow
archive if dlcdn is unreachable.

In [ ]:
import os, re, subprocess, sys, time, urllib.request

t_start = time.time()

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and r.returncode != 0:
        print(r.stdout[-2000:]); print(r.stderr[-2000:])
        raise RuntimeError("command failed: " + cmd)
    return r

# ---------------------------------------------------------------- 1. Java
if not os.path.exists('/usr/lib/jvm'):
    sh('apt-get -qq update')
sh('apt-get -qq install -y openjdk-17-jdk-headless > /dev/null')
java_home = sh("dirname $(dirname $(readlink -f $(which java)))").stdout.strip()
os.environ['JAVA_HOME'] = java_home
print('[ok] java   :', sh('java -version 2>&1 | head -1').stdout.strip())

# ---------------------------------------------------------------- 2. Kafka
def newest_kafka_on_dlcdn():
    '''Ask dlcdn which Kafka it has. Sort numerically: a plain string sort puts
    4.9.0 above 4.10.0, which would pick the wrong one.'''
    html = urllib.request.urlopen(
        'https://dlcdn.apache.org/kafka/', timeout=30).read().decode('utf-8', 'replace')
    vers = sorted({v for v in re.findall(r'href="(\d+\.\d+\.\d+)/"', html)},
                  key=lambda v: [int(p) for p in v.split('.')])
    return vers[-1] if vers else None

KAFKA_DIR = '/content/kafka'
if not os.path.isdir(KAFKA_DIR):
    url = None
    try:
        v = newest_kafka_on_dlcdn()
        if v:
            for scala in ('2.13', '2.12'):
                cand = 'https://dlcdn.apache.org/kafka/%s/kafka_%s-%s.tgz' % (v, scala, v)
                try:
                    urllib.request.urlopen(urllib.request.Request(cand, method='HEAD'),
                                           timeout=20)
                    url = cand; break
                except Exception:
                    continue
    except Exception as e:
        print('     dlcdn unreachable (%s), falling back to the slow archive'
              % type(e).__name__)
    if url is None:
        url = 'https://archive.apache.org/dist/kafka/3.9.1/kafka_2.13-3.9.1.tgz'
        print('     WARNING: using archive.apache.org. This can take many minutes.')
    print('     downloading', url.split('/')[-1], '...')
    sh('wget -q -O /content/kafka.tgz "%s"' % url)
    sh('mkdir -p %s && tar -xzf /content/kafka.tgz -C %s --strip-components=1'
       % (KAFKA_DIR, KAFKA_DIR))
kafka_jar = sh("ls %s/libs/kafka_*.jar | head -1" % KAFKA_DIR).stdout.strip()
KAFKA_VERSION = re.search(r'kafka_[\d.]+-([\d.]+)\.jar', kafka_jar).group(1)
print('[ok] kafka  :', KAFKA_VERSION)

# ---------------------------------------------------------------- 3. Spark
try:
    import pyspark
except ImportError:
    sh('pip install -q pyspark==4.0.1')
    import pyspark
print('[ok] pyspark:', pyspark.__version__)

# ---------------------------------------------------------------- 4. the connector jars
# Spark cannot read Kafka out of the box. The usual advice is
# .config('spark.jars.packages', ...), which makes Spark run Ivy against Maven on
# every single start: minutes, and it sometimes stalls completely. Dropping the
# four jars straight into Spark's own jars folder takes about a second.
JARS = os.path.join(os.path.dirname(pyspark.__file__), 'jars')
SPARK_V = pyspark.__version__
SCALA = '2.13'
need = {
 'spark-sql-kafka-0-10_%s-%s.jar' % (SCALA, SPARK_V):
   'https://repo1.maven.org/maven2/org/apache/spark/spark-sql-kafka-0-10_%s/%s/spark-sql-kafka-0-10_%s-%s.jar' % (SCALA, SPARK_V, SCALA, SPARK_V),
 'spark-token-provider-kafka-0-10_%s-%s.jar' % (SCALA, SPARK_V):
   'https://repo1.maven.org/maven2/org/apache/spark/spark-token-provider-kafka-0-10_%s/%s/spark-token-provider-kafka-0-10_%s-%s.jar' % (SCALA, SPARK_V, SCALA, SPARK_V),
 'kafka-clients-%s.jar' % KAFKA_VERSION:
   'https://repo1.maven.org/maven2/org/apache/kafka/kafka-clients/%s/kafka-clients-%s.jar' % (KAFKA_VERSION, KAFKA_VERSION),
 'commons-pool2-2.12.0.jar':
   'https://repo1.maven.org/maven2/org/apache/commons/commons-pool2/2.12.0/commons-pool2-2.12.0.jar',
}
for name, u in need.items():
    dest = os.path.join(JARS, name)
    if not os.path.exists(dest):
        sh('wget -q -O "%s" "%s"' % (dest, u))
print('[ok] jars   :', len(need), 'connector jars in place')

# ---------------------------------------------------------------- 5. python clients
sh('pip install -q kafka-python-ng pyngrok')
print('[ok] clients: kafka-python-ng, pyngrok')
print()
print('STEP 1 DONE in %.0f seconds' % (time.time() - t_start))

**What you should see.** Five lines beginning `[ok]`, then `STEP 1 DONE` in roughly a
hundred seconds.

If the Kafka download line says it is falling back to `archive.apache.org`, stop the cell
and run it again in a minute. dlcdn is occasionally busy and the archive really is that
slow.

---

# Step 2. Open the tunnel, and do it BEFORE starting Kafka

This is the step people get wrong, so here is why the order matters.

When any Kafka client connects, the first thing it does is ask the broker **"where are
your mailboxes?"**. The broker answers with an address it was told to give out at
startup, called its **advertised address**. The client then throws away the address it
originally dialled and uses the advertised one for everything after that.

So if the broker advertises `localhost`, your laptop will dutifully try to connect to
**your laptop**, find nothing, and hang with an unhelpful timeout. That single behaviour
is the cause of almost every failed Kafka setup you will read about online.

Which means Kafka must be told the tunnel address **when it starts**, so the tunnel has
to exist first. Hence this step comes before Kafka.

**There is nothing to fill in here.** Just run the cell.

In [ ]:
# A tunnel with no account, no password and no credit card.
#
# bore is one small program. It dials out to a shared public server, bore.pub,
# which lends you a port number. Anything sent to bore.pub:thatPort comes back
# down the connection bore already opened, and lands on this Colab machine.
import os, re, subprocess, time, urllib.request

BORE_VERSION = 'v0.6.0'
BORE = '/content/bore'
BORE_LOG = '/content/bore.log'
KAFKA_EXTERNAL_PORT = 9094

if not os.path.exists(BORE):
    url = ('https://github.com/ekzhang/bore/releases/download/{v}/'
           'bore-{v}-x86_64-unknown-linux-musl.tar.gz'.format(v=BORE_VERSION))
    print('downloading bore', BORE_VERSION, '...')
    urllib.request.urlretrieve(url, '/content/bore.tgz')
    subprocess.run('tar -xzf /content/bore.tgz -C /content', shell=True, check=True)
    subprocess.run('chmod +x ' + BORE, shell=True)

ver = subprocess.run(BORE + ' --version', shell=True,
                     capture_output=True, text=True).stdout.strip()
print('[ok] bore   :', ver)

subprocess.run('pkill -f "bore local"', shell=True)   # clear an earlier run
time.sleep(1)
if os.path.exists(BORE_LOG):
    os.remove(BORE_LOG)

# NO_COLOR=1 matters more than it looks. See the comment in wait_for_port().
subprocess.Popen('%s local %d --to bore.pub > %s 2>&1'
                 % (BORE, KAFKA_EXTERNAL_PORT, BORE_LOG),
                 shell=True, env=dict(os.environ, NO_COLOR='1'))


def wait_for_port(path, timeout=45):
    """Read the public port out of bore's log.

    Two things were learned the hard way here, both worth knowing.

    First, bore colours its output even when it is writing to a plain file, so
    the raw bytes are

        remote_port\x1b[0m\x1b[2m=\x1b[0m52013

    and a plain search for 'remote_port=' matches NOTHING, because the name and
    the '=' are not next to each other. NO_COLOR=1 above stops bore doing that.
    The escape stripping below is the belt to that pair of braces.

    Second, the port is not written the instant the process starts. Poll for it
    rather than sleeping a fixed number of seconds and hoping.
    """
    deadline = time.time() + timeout
    clean = ''
    while time.time() < deadline:
        try:
            raw = open(path, encoding='utf-8', errors='replace').read()
        except FileNotFoundError:
            raw = ''
        clean = re.sub(r'\x1b\[[0-9;]*m', '', raw)
        m = (re.search(r'remote_port=(\d+)', clean)
             or re.search(r'listening at bore[.]pub:(\d+)', clean))
        if m:
            return int(m.group(1))
        time.sleep(1)
    raise RuntimeError('bore never reported a port within %d seconds.\n'
                       'bore.pub is a free shared service and is sometimes busy.\n'
                       'Wait a minute and run this cell again.\n\n'
                       'Its log said:\n%s' % (timeout, clean or '(nothing)'))


PUBLIC_HOST = 'bore.pub'
PUBLIC_PORT = wait_for_port(BORE_LOG)

print('tunnel is open')
print()
print('   public address :', '%s:%s' % (PUBLIC_HOST, PUBLIC_PORT))
print('   forwards to    : localhost:%d inside this Colab machine' % KAFKA_EXTERNAL_PORT)
print()
print('Kafka will now be told to advertise exactly that address.')

**What you should see.** A line like `public address : bore.pub:52013`. The port
number is different every time, which is why nothing can be written down in advance.

**If it says bore never reported a port**, bore.pub is a free shared server with no
guarantees and it is occasionally busy or down. Wait a minute and run the cell again. If
it keeps failing, use **`kafka_knime_bridge_A_ngrok.ipynb`** instead, which runs on paid
infrastructure and is more reliable.

**Nothing here needs an account**, so there is no token to paste and nothing to keep
secret.

---

# Step 3. Start Kafka with two front doors

A Kafka broker can listen on several addresses at once, and advertise a different one on
each. We use exactly two:

| Name | Listens on | Advertises | Who uses it |
|---|---|---|---|
| `INTERNAL` | port 9092 | `localhost:9092` | **Spark**, which runs on this same Colab machine |
| `EXTERNAL` | port 9094 | the ngrok address | **KNIME**, on your laptop |

Spark is a local neighbour, so telling it `localhost` is correct and fast. KNIME is a
stranger on the internet, so it must be given the public address. Each client is told the
truth about how to reach the broker *from where it is standing*, and that is the whole
idea.

The other settings are small-cluster housekeeping: one copy of everything, because there
is only one machine.

In [ ]:
import base64, os, re, subprocess, time, glob, uuid

# Colab's JVM prints container-detection warnings to STDOUT before any real
# output. Silence them if this JVM understands the flag. Probed rather than
# assumed, because an option the JVM rejects would stop Kafka starting at all.
if subprocess.run('java -Xlog:disable -version', shell=True,
                  capture_output=True).returncode == 0:
    os.environ['KAFKA_OPTS'] = (
        os.environ.get('KAFKA_OPTS', '') + ' -Xlog:disable').strip()


def drop_jvm_noise(text):
    '''Keep only a tool's real output.

    Even with the flag above, a JVM can still emit lines like

        [0.001s][warning][os,container] Cgroup memory controller path at
        /sys/fs/cgroup seems to have moved to /../../jupyter-children

    Every JVM log line starts with '[', and no Kafka cluster id or topic name
    is allowed to, so dropping those lines is enough.
    '''
    return '\n'.join(l for l in text.splitlines()
                      if l.strip() and not l.lstrip().startswith('['))

LOG_DIR = '/content/kraft-logs'
CFG = '/content/kafka-knime.properties'

subprocess.run('pkill -f kafka.Kafka', shell=True)
time.sleep(3)
subprocess.run('rm -rf %s' % LOG_DIR, shell=True)

props = f'''
process.roles=broker,controller
node.id=1
controller.quorum.voters=1@localhost:9093

listeners=INTERNAL://0.0.0.0:9092,EXTERNAL://0.0.0.0:{KAFKA_EXTERNAL_PORT},CONTROLLER://0.0.0.0:9093
advertised.listeners=INTERNAL://localhost:9092,EXTERNAL://{PUBLIC_HOST}:{PUBLIC_PORT}
listener.security.protocol.map=INTERNAL:PLAINTEXT,EXTERNAL:PLAINTEXT,CONTROLLER:PLAINTEXT
inter.broker.listener.name=INTERNAL
controller.listener.names=CONTROLLER

log.dirs={LOG_DIR}
num.partitions=1
offsets.topic.replication.factor=1
transaction.state.log.replication.factor=1
transaction.state.log.min.isr=1
auto.create.topics.enable=true
'''
open(CFG, 'w').write(props)

print('advertised to Spark  : localhost:9092')
print('advertised to KNIME  : %s:%s' % (PUBLIC_HOST, PUBLIC_PORT))
print()

# Format the storage. Kafka 4.x wants --standalone when the voters are not set;
# we DO set them, so the plain form works. The retry is there in case a future
# version tightens that.
#
# Take the id out of the output rather than using the output as the id. On Colab
# "kafka-storage.sh random-uuid" prints this:
#
#     [0.001s][warning][os,container] Cgroup memory controller path at ...
#     [0.001s][warning][os,container] Cgroup cpu controller path at ...
#     wLE4XIvMROuRNH3MCfPemQ
#
# Pasting all three lines into the next command is what produces the baffling
# "kafka-storage: error: unrecognized arguments: 'Cgroup memory controller'".
raw = subprocess.run('%s/bin/kafka-storage.sh random-uuid' % KAFKA_DIR,
                     shell=True, capture_output=True, text=True).stdout
ids = [w for w in drop_jvm_noise(raw).split()
       if re.fullmatch(r'[A-Za-z0-9_-]{22}', w)]
if ids:
    cid = ids[-1]
else:
    # Nothing usable came back. A Kafka cluster id is only 16 random bytes in
    # url-safe base64 with the padding stripped, so make one here instead. It
    # must not begin with '-' or the next command reads it as a flag.
    cid = '-'
    while cid.startswith('-'):
        cid = base64.urlsafe_b64encode(uuid.uuid4().bytes).decode().rstrip('=')
print('cluster id           :', cid)
fmt = subprocess.run('%s/bin/kafka-storage.sh format -t %s -c %s'
                     % (KAFKA_DIR, cid, CFG), shell=True, capture_output=True, text=True)
if fmt.returncode != 0:
    fmt = subprocess.run('%s/bin/kafka-storage.sh format -t %s -c %s --standalone'
                         % (KAFKA_DIR, cid, CFG), shell=True,
                         capture_output=True, text=True)
    if fmt.returncode != 0:
        print(fmt.stderr[-1500:]); raise RuntimeError('kafka-storage format failed')

subprocess.Popen('%s/bin/kafka-server-start.sh %s > /content/broker.log 2>&1'
                 % (KAFKA_DIR, CFG), shell=True)

# Wait until the broker actually answers, rather than guessing with sleep().
up = False
for i in range(90):
    r = subprocess.run('%s/bin/kafka-topics.sh --bootstrap-server localhost:9092 --list'
                       % KAFKA_DIR, shell=True, capture_output=True, text=True)
    if r.returncode == 0:
        print('broker answered after %d seconds' % i); up = True; break
    time.sleep(1)
if not up:
    print(open('/content/broker.log').read()[-2500:])
    raise RuntimeError('broker did not start, see the log above')

**What you should see.** The two advertised addresses, then
`broker answered after N seconds`, usually under fifteen.

If it never answers, the last part of the broker log is printed for you. The most common
cause is re-running this cell while an old broker is still up, which the `pkill` at the
top is meant to prevent.

---

# Step 4. Create the two mailboxes

A **topic** is a named, append-only list of messages. We need two:

- **`knime-text`** is the inbox. KNIME writes sentences into it.
- **`knime-sentiment`** is the outbox. Spark writes verdicts into it, and KNIME reads them.

Keeping them separate is what makes the design easy to reason about. Nothing ever reads
and writes the same topic, so there is no loop to get confused about.

In [ ]:
IN_TOPIC  = 'knime-text'
OUT_TOPIC = 'knime-sentiment'

for t in (IN_TOPIC, OUT_TOPIC):
    subprocess.run('%s/bin/kafka-topics.sh --bootstrap-server localhost:9092 '
                   '--create --topic %s --partitions 1 --replication-factor 1 '
                   '--if-not-exists' % (KAFKA_DIR, t),
                   shell=True, capture_output=True, text=True)

out = subprocess.run('%s/bin/kafka-topics.sh --bootstrap-server localhost:9092 --list'
                     % KAFKA_DIR, shell=True, capture_output=True, text=True).stdout
print('topics that now exist:')
for line in sorted(drop_jvm_noise(out).split()):
    print('   ', line)

**What you should see.** Both `knime-text` and `knime-sentiment` listed.

---

# Step 5. The word lists that decide the verdict

This is the simplest honest way to score sentiment, and it is the same **lexicon** idea
you may have met in KNIME's text mining nodes: keep a list of positive words and a list
of negative words, then count how many of each appear in a sentence.

The score is

```
score = (positive hits - negative hits) / (positive hits + negative hits)
```

so it always lands between -1 and +1. A sentence with no words from either list scores 0
and is called NEUTRAL.

**This is deliberately not clever.** It cannot read sarcasm, it does not understand
"not good", and it has never heard of your product names. That is a feature for teaching:
every verdict it gives can be explained by pointing at the exact words that caused it,
which is why the answer sent back to KNIME includes those words.

**Edit these lists.** Adding a word your own data cares about is the single easiest way to
improve the result, and it makes the point that a lexicon is a business asset, not code.

In [ ]:
POSITIVE = '''good great excellent amazing wonderful fantastic love loved lovely best
better brilliant superb perfect happy happiest delighted pleased satisfied satisfying
enjoy enjoyed enjoyable recommend recommended outstanding impressive impressed nice
beautiful beautifully smooth fast reliable helpful friendly easy affordable worth
quality comfortable clean fresh solid strong favorite awesome terrific pleasant
exceptional remarkable flawless win wins winning succeed success successful improve
improved improvement gain gains positive glad grateful thank thanks appreciate'''.split()

NEGATIVE = '''bad awful terrible horrible worst worse hate hated dislike poor
disappointing disappointed disappointment useless broken breaks defective faulty
slow sluggish expensive overpriced cheap flimsy fragile uncomfortable dirty stale
weak annoying annoyed frustrating frustrated angry upset unhappy sad regret
waste wasted wasting fail failed failure failing problem problems issue issues
bug bugs buggy crash crashes crashed error errors wrong difficult hard confusing
confused rude unhelpful unreliable never refund return complaint complain'''.split()

print('positive words:', len(POSITIVE))
print('negative words:', len(NEGATIVE))
print()
print('a few of each:', POSITIVE[:6], '...', NEGATIVE[:6])

---

# Step 6. Start Spark watching the inbox

Now the worker. This cell starts a **streaming query**: a job that never finishes. It sits
watching `knime-text`, and the moment a message lands it scores it and writes the answer
to `knime-sentiment`.

Three things worth noticing in the code:

1. **Spark connects to `localhost:9092`**, the INTERNAL door. It is on the same machine as
   the broker, so there is no reason to send its traffic out through the tunnel and back.
2. **`startingOffsets` is `latest`.** The job only looks at messages that arrive after it
   starts, so old test messages do not flood in.
3. **The cell finishes while the job keeps running.** That is normal. `start()` launches
   the query in the background and hands control straight back to you, which is why you
   can carry on to the next cell.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

spark = (SparkSession.builder
         .appName('KnimeSentimentBridge')
         .master('local[2]')
         .config('spark.sql.shuffle.partitions', '2')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print('spark', spark.version)

SCHEMA = StructType([
    StructField('label',         StringType()),
    StructField('score',         DoubleType()),
    StructField('positive_hits', IntegerType()),
    StructField('negative_hits', IntegerType()),
    StructField('matched',       StringType()),
])

pos_set, neg_set = set(POSITIVE), set(NEGATIVE)

def score_text(text):
    if not text:
        return ('NEUTRAL', 0.0, 0, 0, '')
    # keep letters and digits, turn everything else into a space, then split
    words = ''.join(c.lower() if c.isalnum() else ' ' for c in text).split()
    hp = [w for w in words if w in pos_set]
    hn = [w for w in words if w in neg_set]
    p, n = len(hp), len(hn)
    total = p + n
    s = 0.0 if total == 0 else (p - n) / float(total)
    label = 'NEUTRAL' if total == 0 or abs(s) < 0.2 else ('POSITIVE' if s > 0 else 'NEGATIVE')
    matched = ', '.join(['+' + w for w in hp] + ['-' + w for w in hn])
    return (label, round(s, 4), p, n, matched)

sentiment = F.udf(score_text, SCHEMA)

raw = (spark.readStream.format('kafka')
       .option('kafka.bootstrap.servers', 'localhost:9092')   # INTERNAL door
       .option('subscribe', IN_TOPIC)
       .option('startingOffsets', 'latest')
       .load())

answers = (raw
    .selectExpr("CAST(value AS STRING) AS text", 'timestamp')
    .withColumn('s', sentiment(F.col('text')))
    .select(F.col('text'),
            F.col('s.label').alias('label'),
            F.col('s.score').alias('score'),
            F.col('s.positive_hits').alias('positive_hits'),
            F.col('s.negative_hits').alias('negative_hits'),
            F.col('s.matched').alias('matched_words'),
            F.date_format('timestamp', 'yyyy-MM-dd HH:mm:ss').alias('received_at')))

query = (answers
    .select(F.to_json(F.struct('*')).alias('value'))
    .writeStream.format('kafka')
    .option('kafka.bootstrap.servers', 'localhost:9092')
    .option('topic', OUT_TOPIC)
    .option('checkpointLocation', '/content/knime-checkpoint')
    .outputMode('append')
    .start())

# start() hands control back immediately, BEFORE the stream has worked out where
# in the topic to begin reading. Because we asked for 'latest', anything sent in
# that gap is treated as old news and never scored. So wait until Spark reports
# its first completed batch, which only happens once the offsets are settled.
import time as _t
print('waiting for the stream to actually begin reading...')
for _ in range(180):
    if query.lastProgress is not None:
        break
    _t.sleep(1)

if query.lastProgress is None:
    print('WARNING: the stream has not reported progress yet. Give it a moment '
          'and re-run Step 7 if nothing comes back.')
else:
    print('stream reported its first batch, offsets are settled')

print()
print('STREAM IS LIVE:  %s  ->  Spark  ->  %s' % (IN_TOPIC, OUT_TOPIC))
print('leave this running and move to the next cell')

**What you should see.** `spark 4.0.1`, then `STREAM IS LIVE`. The cell stops but the job
does not.

**A warning about re-running this cell.** If you run it twice you will have two jobs
writing the same answers twice. If you need to restart it, run the stop cell at the very
bottom first.

---

# Step 7. Prove it works before you involve KNIME

Always test a pipeline from the inside before adding a second machine to it. If this cell
works, Kafka and Spark are fine, and anything that goes wrong later is a KNIME or network
problem. That one habit will save you hours.

This cell plays both parts: it sends four sentences the way KNIME will, then waits for the
four verdicts to come back.

In [ ]:
import json, time
from kafka import KafkaProducer, KafkaConsumer

TESTS = [
    'I love this product, it works beautifully and the quality is excellent',
    'terrible experience, the thing arrived broken and the support was rude',
    'The package arrived on Tuesday and contained three items',
    'good price but the build is flimsy and it feels cheap',
]

producer = KafkaProducer(bootstrap_servers='localhost:9092',
                         value_serializer=lambda v: v.encode('utf-8'))
for s in TESTS:
    producer.send(IN_TOPIC, s)
producer.flush()
print('sent %d sentences, waiting for Spark...' % len(TESTS))
print()

consumer = KafkaConsumer(OUT_TOPIC, bootstrap_servers='localhost:9092',
                         auto_offset_reset='earliest', consumer_timeout_ms=90000,
                         value_deserializer=lambda b: b.decode('utf-8'))
seen = 0
t0 = time.time()
for msg in consumer:
    r = json.loads(msg.value)
    print('  %-8s score %+.2f   (+%d / -%d)' %
          (r['label'], r['score'], r['positive_hits'], r['negative_hits']))
    print('     text    : %s' % r['text'][:70])
    print('     because : %s' % (r['matched_words'] or '(no words from either list)'))
    print()
    seen += 1
    if seen >= len(TESTS):
        break
consumer.close()
print('%d of %d answers came back in %.0f seconds'
      % (seen, len(TESTS), time.time() - t0))
print('SELF TEST PASSED' if seen == len(TESTS) else 'SELF TEST FAILED, see troubleshooting')

**What you should see**, within a few seconds:

```
  POSITIVE score +1.00   (+4 / -0)
     text    : I love this product, it works beautifully and the quality is ex
     because : +love, +beautifully, +quality, +excellent

  NEGATIVE score -1.00   (+0 / -3)
     because : -terrible, -broken, -rude

  NEUTRAL  score +0.00   (+0 / -0)
     because : (no words from either list)

  NEGATIVE score -0.33   (+1 / -2)
     because : +good, -flimsy, -cheap
```

Look at the last one. "good price but the build is flimsy and it feels cheap" is a mixed
review, and the score lands at -0.33 rather than at either extreme. One positive word
against two negative ones. That is the arithmetic being honest about a genuinely mixed
sentence.

The third one is the useful failure. There is nothing emotional in it, so the lexicon has
nothing to say, and it correctly refuses to guess.

---

# Step 8. The address to paste into KNIME

Run this cell and copy the line it prints. **That single line is the only thing KNIME
needs from this notebook.**

It changes every time you restart the tunnel, so copy it fresh each session.

In [ ]:
print()
print('=' * 62)
print('  PASTE THIS INTO THE KNIME "Kafka Connector" NODE')
print('=' * 62)
print()
print('    %s:%s' % (PUBLIC_HOST, PUBLIC_PORT))
print()
print('=' * 62)
print()
print('  Topic KNIME PRODUCER writes to :  %s' % IN_TOPIC)
print('  Topic KNIME CONSUMER reads from:  %s' % OUT_TOPIC)
print()
print('  Keep this Colab tab open and the runtime awake while you use KNIME.')

---

# Step 9. Watch it happen while you use KNIME

Go to KNIME now and follow the instruction document. Come back and run this cell to watch
messages arrive in real time. It listens for two minutes, then stops.

Run it, then switch to KNIME and execute your Producer node. You should see the sentence
appear here almost instantly.

In [ ]:
from kafka import KafkaConsumer
import json, time

WATCH_SECONDS = 120

watch = KafkaConsumer(OUT_TOPIC, bootstrap_servers='localhost:9092',
                      auto_offset_reset='latest', consumer_timeout_ms=WATCH_SECONDS * 1000,
                      value_deserializer=lambda b: b.decode('utf-8'))
print('watching %s for %d seconds. Send something from KNIME now.' % (OUT_TOPIC, WATCH_SECONDS))
print()
n = 0
for msg in watch:
    r = json.loads(msg.value)
    n += 1
    print('%2d. %-8s %+.2f  "%s"' % (n, r['label'], r['score'], r['text'][:52]))
watch.close()
print()
print('stopped watching. %d message(s) seen.' % n)
if n == 0:
    print('Nothing arrived. Check the troubleshooting table at the bottom.')

---

# Step 10. Stopping cleanly

Run this when you are finished. Colab will also throw everything away on its own when the
runtime ends, so this is tidiness rather than necessity.

In [ ]:
try:
    query.stop(); print('spark stream stopped')
except Exception as e:
    print('stream already stopped (%s)' % type(e).__name__)
subprocess.run('pkill -f "bore local"', shell=True)
print('tunnel closed')
subprocess.run('pkill -f kafka.Kafka', shell=True)
print('kafka stopped')

---

# If something went wrong

| What you saw | What it means and what to do |
|---|---|
| KNIME says **timeout** or **connection refused** | Almost always the advertised address. Re-run Step 3 *after* Step 2, never before, and make sure the address in KNIME matches what Step 8 printed exactly. |
| KNIME connects but **no messages ever arrive** | The topic name is misspelled in KNIME. The Producer must write to `knime-text` and the Consumer must read `knime-sentiment`. They are not interchangeable. |
| **bore never reported a port** | bore.pub is a free shared server and is sometimes busy. Wait a minute and run Step 2 again, or switch to the ngrok notebook. |
| **The tunnel dies partway through class** | bore.pub gives no uptime promise. Re-run Step 2, then Step 3, because the port will have changed. |
| **KNIME worked, then stopped** | Check Step 2's log. If bore reconnected it will have a different port, and Kafka is still advertising the old one. Re-run Step 2 and Step 3. |
| **Step 7 works but KNIME does not** | Kafka and Spark are fine, so it is the network or KNIME. Check the address, and check that your laptop is not on a network that blocks unusual outbound ports, which some campus wifi does. |
| **Answers come back twice** | You ran Step 6 more than once, so two streams are running. Run Step 10, then Step 6 again, once. |
| **Everything stops after a while** | Colab disconnects idle runtimes. Keep the browser tab visible, and re-run from Step 2 if it drops. The bore port will be different. |
| `UnsupportedVersionError` | A Kafka client much older than the broker. Re-run Step 1. |
| **`kafka-storage: error: unrecognized arguments: 'Cgroup ...'`** | Colab's JVM printed container warnings onto stdout and they ended up inside the cluster id. Step 3 now strips them. If you see it again, re-run Step 3 as it stands now. |

---

# If you want the more reliable tunnel instead

Open **`kafka_knime_bridge_A_ngrok.ipynb`**. It uses ngrok, which wants a free
account and a credit card for verification. ngrok does not charge the card; it is
an anti-abuse check they added after their free tunnels were used for malware.

The two notebooks are identical from Step 3 onwards.

---

# What to take away from this lab

1. **A message broker lets two programs cooperate without knowing each other.** KNIME has
   no idea Spark exists. Spark has no idea KNIME exists. Both only know Kafka. You could
   replace either side tomorrow and the other would not notice.

2. **`advertised.listeners` is the idea worth remembering.** A server behind a tunnel, a
   load balancer or a container has to tell clients an address that works *from where the
   client is standing*, not from where the server is. The same problem appears in Docker,
   Kubernetes and every cloud deployment you will ever do.

3. **Test from the inside first.** Step 7 exists so that when KNIME misbehaves you already
   know the pipeline is not at fault.

4. **The clever part is not the model.** The sentiment scorer is forty lines of word
   counting. The engineering around it, getting text in, getting answers out, and letting
   two machines meet in the middle, is the part that took the work.